## Chicago Bronze to Silver
Reads the latest file_location of the table from the metadata table,
loads the Parquet snapshot, and overwrites the corresponding Silver Delta table.

In [0]:
%pip install databricks-labs-dqx
# method to restart
%restart_python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.1/775.1 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 606.1/606.1 kB 15.8 MB/s eta 0:00:00
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 6.0.2
    Not uninstalling pyyaml at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-8c1eb8fc-bbef-4aff-9839-eebe5db4e78e
    Can't uninstall 'PyYAML'. No files were found to uninstall.
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Not uninstalling protobuf at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-8c1eb8fc-bbef-4aff-9839-eebe5db4e78e
    Can't uninstall 'protobuf'. No files were found to uninstall.
  Attempting uninstall: dat

In [0]:
catalog_name  = dbutils.widgets.get("catalog_name")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")

print(f"catalog_name  : {catalog_name}")
print(f"bronze_schema : {bronze_schema}")
print(f"silver_schema : {silver_schema}")

catalog_name  : workspace
bronze_schema : final_project_bronze
silver_schema : final_project_silver


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DateType, LongType, IntegerType
from datetime import datetime, timezone

run_ts = datetime.now(timezone.utc)

### Get Latest File Location per Table from Child Metadata

In [0]:
child_df = spark.table(f"{catalog_name}.{bronze_schema}.pipeline_metadata_child")

# Pick the most recent SUCCESS run per table
latest_df = (
    child_df
    .filter("status = 'SUCCESS'")
    .groupBy("table_name")
    .agg(F.max_by("file_location", "execution_time").alias("file_location"))
)

latest_rows = latest_df.collect()
print(f"Tables to promote to Silver: {[r.table_name for r in latest_rows]}")

Tables to promote to Silver: ['Food_Inspections_Chicago', 'Food_Inspections_Dallas']


### Load Bronze Parquet → DQX → Transformations → Write Silver Delta (overwrite)

In [0]:
# load chicago file
table_name = latest_rows[0].table_name
file_loc   = latest_rows[0].file_location

print(f"  Processing: {table_name}")
print(f"  Source: {file_loc}")

chicago_df = spark.read.parquet(file_loc)

  Processing: Food_Inspections_Chicago
  Source: /Volumes/workspace/final_project_bronze/food_inspection/food_inspections_chicago/2026/04/08/food_inspections_chicago_20260408_230522.parquet


In [0]:
chicago_df = chicago_df\
    .withColumnRenamed("Inspection ID", "inspection_id")\
    .withColumnRenamed("DBA Name", "establishment_name")\
    .withColumnRenamed("AKA Name", "aka_establishment_name")\
    .withColumnRenamed("License #", "license_number")\
    .withColumnRenamed("Facility Type", "facility_type")\
    .withColumnRenamed("Risk", "risk")\
    .withColumnRenamed("Address", "street_address")\
    .withColumnRenamed("City", "city")\
    .withColumnRenamed("Zip", "zip_code")\
    .withColumnRenamed("Inspection Date", "inspection_date")\
    .withColumnRenamed("Inspection Type", "inspection_type")\
    .withColumnRenamed("Results", "inspection_result")\
    .withColumnRenamed("Violations", "violations")\
    .withColumnRenamed("Latitude", "latitude")\
    .withColumnRenamed("Longitude", "longitude")\
    .withColumnRenamed("Location", "location")\
    .drop("State")

In [0]:
chicago_df = chicago_df\
                        .withColumn("latitude", F.col("latitude").try_cast("double"))\
                        .withColumn("longitude", F.col("longitude").try_cast("double"))\
                        .withColumn("zip_code", F.col("zip_code").cast("string"))

chicago_df = chicago_df.withColumn(
    "valid_geo",
    F.col("latitude").isNotNull() & F.col("longitude").isNotNull()
)

chicago_df = chicago_df \
    .withColumn("latitude", F.when(F.col("valid_geo"), F.col("latitude"))) \
    .withColumn("longitude", F.when(F.col("valid_geo"), F.col("longitude"))) \
    .withColumn("location", F.when(F.col("valid_geo"), F.col("location"))) \
    .drop("valid_geo")

In [0]:
chicago_df = (chicago_df
                .withColumn("latitude",
                    F.when(F.col("location").isNull(), None).otherwise(F.col("latitude")))
                .withColumn("longitude",
                    F.when(F.col("location").isNull(), None).otherwise(F.col("longitude")))
            )

### Data Profiling

### Databricks DQX library modules
- **WorkspaceClient** Authenticate the Databricks SDK for Python with your Databricks account or workspace to perform data quality checking with DQX
- **DQEngine** Run the data quality checks

In [0]:
from databricks.labs.dqx.engine import DQEngine
from databricks.sdk import WorkspaceClient

In [0]:
# The engine requires a Databricks workspace client for authentication and interaction with the Databricks workspace
ws = WorkspaceClient()

non_null_cols = ["establishment_name", "inspection_date", "inspection_type", "zip_code", "inspection_result", "violations"]

non_null_checks = [
    {
        "criticality": "error",
        "name": f"{col}_is_null",
        "check": {
            "function": "sql_expression",
            "arguments": {
                "expression": f"{col} is not null",
                "msg": f"{col} must not be null",
            },
        },
    }
    for col in non_null_cols
]

# apply the checks generated based on profile to validate
dqengine = DQEngine(ws)
valid_chicago_df, invalid_chicago_df = dqengine.apply_checks_by_metadata_and_split(chicago_df, non_null_checks)

In [0]:
passed_count = valid_chicago_df.count()
failed_count = invalid_chicago_df.count()
print(f"  Passed: {passed_count} | Quarantined: {failed_count}")

# Write quarantine records
if failed_count > 0:
    data_cols = [c for c in invalid_chicago_df.columns if not c.startswith("_")]
    quarantine_out = (
        invalid_chicago_df
        .withColumn("table_name",     F.lit(table_name))
        .withColumn("run_timestamp",  F.lit(run_ts).cast(TimestampType()))
        .withColumn("_error_details", F.to_json(F.col("_errors")))
        .withColumn("_row_data",      F.to_json(F.struct(*[F.col(c) for c in data_cols])))
        .select("table_name", "run_timestamp", "_error_details", "_row_data")
    )
    quarantine_out.write.format("delta").mode("append").saveAsTable(
        f"{catalog_name}.{silver_schema}.chicago_quarantine"
    )
    print(f"{failed_count} records written to chicago quarantine.")

dqx_log_rows = []

dqx_log_rows.append((
    table_name,
    run_ts,
    passed_count,
    failed_count,
    run_ts.date()
))

  Passed: 221873 | Quarantined: 86484
86484 records written to chicago quarantine.


### Write DQX Execution Log

In [0]:
log_schema = StructType([
    StructField("table_name",     StringType(),    True),
    StructField("run_timestamp",  TimestampType(), True),
    StructField("passed_records", LongType(),      True),
    StructField("failed_records", LongType(),      True),
    StructField("created_date",   DateType(),      True),
])

log_df = spark.createDataFrame(dqx_log_rows, log_schema)
log_df.write.format("delta").mode("append").saveAsTable(
    f"{catalog_name}.{silver_schema}.dqx_execution_log"
)

print("\nDQX execution log written.")
display(log_df)


DQX execution log written.


table_name,run_timestamp,passed_records,failed_records,created_date
Food_Inspections_Chicago,2026-04-11T15:51:14.984Z,221873,86484,2026-04-11


In [0]:
# regex cleaning
# clean up string columns
for col_name, dtype in valid_chicago_df.dtypes:
    if dtype == "string":
        valid_chicago_df = valid_chicago_df\
                                .withColumn(col_name, F.trim(F.col(col_name)))\
                                .withColumn(col_name, F.upper(F.col(col_name)))

# remove special characters from name columns
cols = ["establishment_name", "aka_establishment_name", "street_address"]

for c in cols:
    valid_chicago_df = valid_chicago_df\
                        .withColumn(c, F.regexp_replace(F.col(c), r"[\\.,']", ""))

# remove double quotes from violations column
valid_chicago_df = valid_chicago_df.withColumn(
    "violations",
    F.regexp_replace(F.col("violations"), r'^"|"$', "")
)
                    

In [0]:
# create establishment id
valid_chicago_df = valid_chicago_df.withColumn(
    "establishment_id", F.sha2( F.concat_ws("||",
            F.coalesce(F.col("establishment_name"), F.lit("")),
            F.coalesce(F.col("street_address"), F.lit("")),
            F.coalesce(F.col("city"), F.lit("")),
            F.coalesce(F.col("zip_code"), F.lit(""))
        ), 256
    )
)

In [0]:
# create inspection score column
valid_chicago_df = valid_chicago_df\
                        .withColumn("inspection_score",
                         F.when(F.col("inspection_result") == "PASS", 90)
                         .when(F.col("inspection_result") == "PASS W/ CONDITIONS", 80)
                         .when(F.col("inspection_result") == "FAIL", 70)
                         .when(F.col("inspection_result") == "NO ENTRY", 00)
                         .otherwise(None)
                    )

In [0]:
# explode the df to get violation level grain
valid_chicago_df = valid_chicago_df.withColumn("violation_array", F.split(F.col("violations"), r"\s*\|\s*"))

valid_chicago_df = valid_chicago_df.withColumn("exploded_violation", F.explode("violation_array"))

valid_chicago_df = valid_chicago_df \
    .withColumn(
        "violation_code",
        F.regexp_extract(F.col("exploded_violation"), r"^(\d+)\.", 1)
    ) \
    .withColumn(
        "violation_description",
        F.regexp_extract(F.col("exploded_violation"), r"^\d+\.\s*(.*?)\s*-\s*COMMENTS:", 1)
    ) \
    .withColumn(
        "violation_comments",
        F.regexp_extract(F.col("exploded_violation"), r"COMMENTS:\s*(.*)", 1)
    )

In [0]:
valid_chicago_df = valid_chicago_df \
                    .drop(
                        "violations",
                        "violation_array",
                        "exploded_violation")
                       

In [0]:
valid_chicago_df = valid_chicago_df.groupBy(
    'inspection_id',
    'establishment_name',
    'aka_establishment_name',
    'license_number',
    'facility_type',
    'risk',
    'street_address',
    'city',
    'zip_code',
    'inspection_date',
    'inspection_type',
    'inspection_result',
    'latitude',
    'longitude',
    'location',
    'establishment_id',
    'inspection_score',
    'violation_code',
    'violation_description'
).agg(
    F.concat_ws(
        " | ",
        F.sort_array(F.collect_set("violation_comments"))
    ).alias("violation_comments")
)

In [0]:
valid_chicago_df = valid_chicago_df.dropDuplicates()

valid_chicago_df = valid_chicago_df\
                        .withColumn("source_table", F.lit(table_name))\
                        .withColumn("load_date", F.lit(run_ts))


In [0]:
valid_chicago_df.columns

['inspection_id',
 'establishment_name',
 'aka_establishment_name',
 'license_number',
 'facility_type',
 'risk',
 'street_address',
 'city',
 'zip_code',
 'inspection_date',
 'inspection_type',
 'inspection_result',
 'latitude',
 'longitude',
 'location',
 'establishment_id',
 'inspection_score',
 'violation_code',
 'violation_description',
 'violation_comments',
 'source_table',
 'load_date']

In [0]:
# Write to Silver
(
    valid_chicago_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{silver_schema}.chicago_food_inspection")
)
print(f"  Silver table chicago_food_inspection written.")

  Silver table chicago_food_inspection written.
